# 1. Configurar PySpark e iniciar sesión

In [122]:
# Importamos la librería de apache spark
import pyspark as ps
from pyspark.sql import SparkSession # Importamos SparkSesion
from pyspark.sql.types import IntegerType, StringType, FloatType, DoubleType, DateType, TimestampType # Importamos tipos de datos
from datetime import datetime
import time

print(f'Versión de Spark: {ps.__version__}')

Versión de Spark: 3.5.1


In [2]:
# Primero, creamos una instancia de SparkContext
# Usamos SparkSession.builder para obtener un SparkContext
spark = SparkSession.builder \
    .appName("Rover_RDD") \
    .getOrCreate()

# 3. Crear el RDD a partir de los datos

Para esto voy a emplear el csv de un proyecto anterior del máster, en el que planteé un sistema de logística para el sistema solar.

Las columnas en este dataframe son:


*   ID Envio: Identificador del pedido, INT.
*   Origen: Nombre del lugar de origen, STRING.
*   Destino: Nombre del lugar de destino, STRING.
*   Tipo de carga: Designación del elemento transportado, STRING.
*   Masa (Kg): Masa del pedido, INT.
*   Tiempo de entrega: Tiempo estimado de entrega en días, INT.
*   Coste estimado (€): Coste estimado del pedido previo a contratación, FLOAT.
*   Estado: Prioridad del envío, STRING.
*   Estado de envío: Estado actual del envío, STRING.
*   Manifiesto: Identificador del manifiesto en el que se refleja el envío, STRING.
*   Consumo combustible (L): Litros de combustible consumidos imputables al envío, FLOAT.
*   Coordenadas origen: Coordenadas en base galáctica del punto de origen, FLOAT.
*   Coordenadas destino: Coordenadas en base galáctica del punto de destino, FLOAT.
*   Fecha envío: Fecha cuando se realiza el envío desde el lugar de origen, TIMESTAMP.
*   ID_Nave: Identificador de la nave a la que se le asigna el envío, STRING.



In [3]:
data_path = '/content/logistica_sistema_solar_TSC.csv'

In [4]:
# Leer el archivo CSV con Spark
lines = spark.sparkContext.textFile(data_path)

df = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .load(data_path)

# Mostrar el DataFrame
df.show()

+--------+--------------------+--------------------+--------------------+-------+-------------------+--------------------+------+------------+-----------+---------------------+------------------+-------------------+-----------+-------+
|ID_Envio|              Origen|             Destino|          Tipo_Carga|Masa_kg|Tiempo_Entrega_días|Costo_Estimado_Euros|Estado|Estado_Envio| Manifiesto|Consumo_Combustible_L|Coordenadas_Origen|Coordenadas_Destino|Fecha_Envio|ID_Nave|
+--------+--------------------+--------------------+--------------------+-------+-------------------+--------------------+------+------------+-----------+---------------------+------------------+-------------------+-----------+-------+
|       1|Estación Orbital ...|              Europa|   Equipo Científico|    808|                 17|             49782.8|  Alta|    Demorado|MF-6C698688|              1079.16| -95.2638,-49.4028|  -126.4673,30.8775| 2125-01-04|TSC-001|
|       2|      Estación Ceres|               Marte|Alim

Exploramos el RDD

In [38]:
# Mostrar el esquema del DataFrame
df.printSchema()
schema = df.schema

# Contar el número de filas en el DataFrame
row_count = df.count()
print(f"El DataFrame tiene {row_count} filas.")

root
 |-- ID_Envio: integer (nullable = true)
 |-- Origen: string (nullable = true)
 |-- Destino: string (nullable = true)
 |-- Tipo_Carga: string (nullable = true)
 |-- Masa_kg: integer (nullable = true)
 |-- Tiempo_Entrega_días: integer (nullable = true)
 |-- Costo_Estimado_Euros: double (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Estado_Envio: string (nullable = true)
 |-- Manifiesto: string (nullable = true)
 |-- Consumo_Combustible_L: double (nullable = true)
 |-- Coordenadas_Origen: string (nullable = true)
 |-- Coordenadas_Destino: string (nullable = true)
 |-- Fecha_Envio: date (nullable = true)
 |-- ID_Nave: string (nullable = true)

El DataFrame tiene 100 filas.


# 4. Preparando los datos para análisis

Tal como está almacenado ahora mismo el RDD, se almacena línea a línea, vamos a comprobarlo extrayendo un elemento y comprobando su contenido.

In [12]:
# Obtener el segundo elemento del RDD usando take()
# take(n) devuelve una lista Python con los primeros n elementos del RDD
header = lines.take(2)[0]
first_line = lines.take(2)[1]

# Imprimir el segundo elemento
print(first_line)

001,Estación Orbital Saturno,Europa,Equipo Científico,808,17,49782.8,Alta,Demorado,MF-6C698688,1079.16,"-95.2638,-49.4028","-126.4673,30.8775",2125-01-04,TSC-001


In [164]:
# Vamos a solventar el problema del tratamiento de las coordenadas
# Para ello vamos a crear una clase específica

class coordinates():
    # Definimos la función generadora, que requiere las dos coordenadas
    def __init__(self, x, y):
        self.x = x
        self.y = y

    # Redefinimos la función que imprime las coordenadas
    def __str__(self):
        return f'({self.x}, {self.y})'

    # Definimos la función para determinar el cuadrante
    def cuadrante(self):

        if self.x >= 0 and self.y >= 0:
            return 1
        elif self.x < 0 and self.y >= 0:
            return 2
        elif self.x < 0 and self.y < 0:
            return 3
        else:
            return 4

In [177]:
# Vamos a definir una función auxiliar para tratar los datos de las coordenadas
# El problema es que las toma como dos columnas al estar separadas por comas.
# Vamos a tomar los datos de la coordenada X e Y del split y convertirlos en clase coordenadas
def remake_coordinates(fields):
    new_f_line = []
    x = 0
    y = 0
    for i, item in enumerate(fields):
        if i == 11 or i == 13:
            x = float(item[1:])
        elif i == 12 or i == 14:
            y = float(item[:-1])
            new_f_line.append(coordinates(x, y))
        else:
            new_f_line.append(item)

    return new_f_line

La siguiente función servirá para el mapeado del RDD original, actuando sobre cada línea que, como hemos visto, contiene el dato de cada columna separado por una coma. Esta función tomará linea a linea, separará los datos por columnas y ajustará el tipo de acuerdo a un esquema apropiado.

Además, se le ha incluido una variable para el uso de funciones auxiliares. En este caso vamos a emplear la función que hemos creado en el paso previo para poder tratar adecuadamente las coordenadas, dado que aparecen separadas por comas y no queremos que nos las divida, creando columnas extra.

In [166]:
# Probamos otra función que mantenga los tipos de datos de acuerdo al esquema
def parse_line_with_types(line, schema, aux_funcion = None):
    fields = line.split(',')
    # Usar el split sin más tiene el problema de separar las coordenadas, vamos a unirlas
    # Añadimos el bloque de código para usar la función auxiliar, si precisa.
    if aux_funcion is not None:
        new_f_line = aux_funcion(fields)
    else:
        new_f_line = fields

    # Comprobamos que el esquema pertenece a los datos
    if len(new_f_line) != len(schema):
        # Aviso en caso de que sea incorrecto
        print(f'Error: Esquema incorrecto')
        return new_f_line

    # Inicializamos una nueva lista donde guardaremos los datos tipados
    parsed_fields = []
    for i, field in enumerate(new_f_line):
          field_name = schema[i].name
          field_type = schema[i].dataType

          if isinstance(field_type, IntegerType): #Convertimos enteros
              parsed_fields.append(int(field))
          elif isinstance(field_type, FloatType) or isinstance(field_type, DoubleType): #Convertimos flotantes
              parsed_fields.append(float(field))
          elif isinstance(field_type, DateType): # Convertimos fechas formate Date
              parsed_fields.append(datetime.strptime(field, '%Y-%m-%d'))
          elif isinstance(field_type, TimestampType): # Convertimos datastamps
              parsed_fields.append(datetime.strptime(field, '%Y-%m-%d %H:%M:%S'))
          else:  # Si no es ninguno de los casos anteriores, guardamos como el tipo que tiene
              parsed_fields.append(field)

    return parsed_fields

Las dos funciones anteriores podrían estar integradas en una sola función, pero de este modo la función para el tratamiento de datos es más general y puede ser empleadas en otros casos.

In [167]:
# Creamos el RDD con los datos por columnas manteniendo tipos

# Extraemos la cabecera que contiene las columnas
header = lines.first()
data_lines = lines.filter(lambda line: line != header)

# Aplicamos la transformación a las líneas con datos
parsed = data_lines.map(lambda line: parse_line_with_types(line, schema, remake_coordinates))

In [168]:
# Vamos a mostrar ahora el contenido del nuevo RDD
print(parsed.take(2))
print(type(parsed.take(2)[0][-2]))

[[1, 'Estación Orbital Saturno', 'Europa', 'Equipo Científico', 808, 17, 49782.8, 'Alta', 'Demorado', 'MF-6C698688', 1079.16, <__main__.coordinates object at 0x799a13db0fd0>, <__main__.coordinates object at 0x799a1822b050>, datetime.datetime(2125, 1, 4, 0, 0), 'TSC-001'], [2, 'Estación Ceres', 'Marte', 'Alimentos Procesados', 1154, 380, 13639.95, 'Media', 'Programado', 'MF-474C9A62', 621.69, <__main__.coordinates object at 0x799a13f4bc90>, <__main__.coordinates object at 0x799a18228ed0>, datetime.datetime(2125, 4, 30, 0, 0), 'TSC-010']]
<class 'datetime.datetime'>


Como vemos, los datos se almacenan como string en los RDD, por lo que tendremos que tenerlo en cuenta antes de hacer transformaciones.

# 5. Realizamos transformaciones sobre los datos


En primer lugar vamos a hacer una comprobación de los envíos de alta prioridad que se han retrasado, para ello tendremos que hacer comprobaciones booleanas en dos columnas y mostrar los resultados.

In [169]:
# Vamos a mostrar todos los envíos demorados de prioridad alta
# Comprobamos que se cumplan ambas condiciones y lo almacenamos en un RDD
HP_Delayed = parsed.filter(lambda x: 'Alta' in x[7] and 'Demorado' in x[8])
total_delayed = HP_Delayed.count()

# Mostramos los resultados
print(f'Total de envíos de alta prioridad retrasados: {total_delayed}')
for line in HP_Delayed.collect():
    print(f'ID de envio: {line[0]}, manifiesto: {line[9]}')

Total de envíos de alta prioridad retrasados: 9
ID de envio: 1, manifiesto: MF-6C698688
ID de envio: 33, manifiesto: MF-646BA63C
ID de envio: 40, manifiesto: MF-C7CC45F5
ID de envio: 67, manifiesto: MF-D84E7C86
ID de envio: 69, manifiesto: MF-C0A6782A
ID de envio: 75, manifiesto: MF-CDD02959
ID de envio: 76, manifiesto: MF-31A85EA9
ID de envio: 90, manifiesto: MF-E557EE0A
ID de envio: 96, manifiesto: MF-C90670E8


Como segunda visualización, vamos a mostrar el total de envíos efectuados desde la Tierra, ordenados por el coste estimado de envío.

In [103]:
# Vamos a visualizar todos los envíos que salen de la Tierra
Sended_Earth = parsed.filter(lambda x: 'Tierra' in x[1])
total_earth = Sended_Earth.count()

#Los ordenamos por coste estimado
Sended_E_bycost = Sended_Earth.sortBy(lambda x: x[6], ascending=False)

# Mostramos los resultados
print(f'Total de envíos que salen de la Tierra: {total_earth}')
for line in Sended_E_bycost.collect():
    print(f'ID de envio: {line[0]}, Coste estimado: {line[6]} €')

Total de envíos que salen de la Tierra: 6
ID de envio: 11, Coste estimado: 47668.0 €
ID de envio: 75, Coste estimado: 42497.45 €
ID de envio: 45, Coste estimado: 40743.05 €
ID de envio: 99, Coste estimado: 35745.9 €
ID de envio: 88, Coste estimado: 16519.75 €
ID de envio: 21, Coste estimado: 12711.75 €


Ahora vamos a comprobar el total de envíos realizado en Marzo de 2125 desde un cuerpo celeste que en estas fechas se encontraba en el segundo cuadrante galáctico.

In [173]:
# Vamos a extraer los envíos efectuados en Marzo de 2125 desde el segundo cuadrante galáctico
# Vamos a emplear las funciones de la nueva clase coordenadas

# Procesamos los datos
Sended_March = parsed.filter(lambda x: x[13].strftime('%Y-%m') == '2125-03')
sec_quadrant = Sended_March.filter(lambda x: x[11].cuadrante() == 2)
total_march = sec_quadrant.count()

# Mostramos los resultados
print(f'Total de envíos efectuados en Marzo de 2125 en el segundo cuadrante: {total_march}')
for line in sec_quadrant.collect():
    print(f'ID de envio: {line[0]}, Fecha de envío: {line[13].strftime("%Y-%m-%d")}, origen: {line[1]}, coordenadas: {line[11]}')



Total de envíos efectuados en Marzo de 2125 en el segundo cuadrante: 1
ID de envio: 79, Fecha de envío: 2125-03-20, origen: Estación Ceres, coordenadas: (-21.1455, 21.3859)


# 6 Comparativa de tiempo de ejecución con persistencia

Ahora vamos a comparar el tiempo de ejecución para el procesamiento de los datos de forma que cada RDD necesita persistirse, con respecto al caso anterior donde se efectuaba en memoria.

In [176]:
# Iniciamos el tiempo
init_time = time.time()

# Ejecutamos una transformación de datos
# Vamos a calcular la masa total transportada por la nave TSC-001
mass_TSC_001 = parsed.filter(lambda x: x[14] == 'TSC-001').map(lambda x: (x[0], x[4]))
total_mass = mass_TSC_001.reduce(lambda x, y: (None, x[1] + y[1]))[1]

# Mostramos el resultado
print(f'La masa total transportada por la nave TSC-001 es: {total_mass} kg')

# Mostramos todos los envíos
for envio in mass_TSC_001.collect():
    print(f'ID Envio: {envio[0]}, Masa: {envio[1]} Kg')

# Finalizamos el temporizador
end_time = time.time()
elapsed_time = end_time - init_time
print(f'Tiempo de ejecución: {elapsed_time} segundos')

La masa total transportada por la nave TSC-001 es: 12431 kg
ID Envio: 1, Masa: 808 Kg
ID Envio: 16, Masa: 726 Kg
ID Envio: 19, Masa: 1230 Kg
ID Envio: 20, Masa: 1789 Kg
ID Envio: 27, Masa: 385 Kg
ID Envio: 41, Masa: 208 Kg
ID Envio: 56, Masa: 1045 Kg
ID Envio: 60, Masa: 1081 Kg
ID Envio: 69, Masa: 549 Kg
ID Envio: 76, Masa: 1666 Kg
ID Envio: 81, Masa: 470 Kg
ID Envio: 93, Masa: 618 Kg
ID Envio: 100, Masa: 1856 Kg
Tiempo de ejecución: 0.5202507972717285 segundos


In [175]:
# Repetimos el procedimiento anterior, pero con persistencia
# Iniciamos el tiempo
init_time = time.time()

# Ejecutamos una transformación de datos
# Vamos a calcular la masa total transportada por la nave TSC-001
mass_TSC_001 = parsed.filter(lambda x: x[14] == 'TSC-001').map(lambda x: (x[0], x[4])).persist()
total_mass = mass_TSC_001.reduce(lambda x, y: (None, x[1] + y[1]))[1]

# Mostramos el resultado
print(f'La masa total transportada por la nave TSC-001 es: {total_mass} kg')

# Mostramos todos los envíos
for envio in mass_TSC_001.collect():
    print(f'ID Envio: {envio[0]}, Masa: {envio[1]} Kg')

# Finalizamos el temporizador
end_time = time.time()
elapsed_time = end_time - init_time
print(f'Tiempo de ejecución: {elapsed_time} segundos')

La masa total transportada por la nave TSC-001 es: 12431 kg
ID Envio: 1, Masa: 808 Kg
ID Envio: 16, Masa: 726 Kg
ID Envio: 19, Masa: 1230 Kg
ID Envio: 20, Masa: 1789 Kg
ID Envio: 27, Masa: 385 Kg
ID Envio: 41, Masa: 208 Kg
ID Envio: 56, Masa: 1045 Kg
ID Envio: 60, Masa: 1081 Kg
ID Envio: 69, Masa: 549 Kg
ID Envio: 76, Masa: 1666 Kg
ID Envio: 81, Masa: 470 Kg
ID Envio: 93, Masa: 618 Kg
ID Envio: 100, Masa: 1856 Kg
Tiempo de ejecución: 0.8606438636779785 segundos


Al añadir persistencia al RDD donde se realiza el filtrado y el mapeo, se aprecia cierto aumento en el tiempo de ejecución. Sin embargo al ejecutarlo en Colab el tiempo total es bastante variable entre diferentes ejecuciones, por lo que es dificil precisar cuál es la diferencia.